# 10 · Análisis en Python: el mismo problema con pandas y PySpark

Este notebook resuelve **8 preguntas de negocio** con **pandas** y con **PySpark**, una al lado de la otra. Son las mismas preguntas del notebook [`09_SQL_Analytics`](../analytics/09_SQL_Analytics), así que cada problema queda resuelto de tres formas: SQL, pandas y PySpark.

**Cómo usarlo:** conecta el notebook a **Serverless** y presiona **Run all**. Cada celda muestra su tabla de resultados con `display()`.

| # | Pregunta | SQL | pandas | PySpark |
|---|---|---|---|---|
| 1 | Tasa promedio del dólar por año | CTE + `GROUP BY` | `groupby().agg()` | `groupBy().agg()` |
| 2 | Gasolina premium semana a semana | `LAG` | `shift()` | `F.lag().over()` |
| 3 | Último precio de cada combustible | `ROW_NUMBER` | `sort_values().groupby().head(1)` | `F.row_number().over()` |
| 4 | Ranking de pérdidas por distribuidora | `RANK` / `DENSE_RANK` | `rank(method="min"/"dense")` | `F.rank()` / `F.dense_rank()` |
| 5 | Participación % de cada distribuidora | `SUM() OVER ()` | `x / x.sum()` | `F.sum().over(Window.partitionBy())` |
| 6 | Energía acumulada en 2026 | `SUM … UNBOUNDED PRECEDING` | `cumsum()` | `F.sum().over(… rowsBetween)` |
| 7 | Promedio móvil de 3 meses de la inflación | `AVG … ROWS BETWEEN` | `rolling(3).mean()` | `F.avg().over(… rowsBetween(-2, 0))` |
| 8 | 25 % de productos agrícolas que más subió | `NTILE(4)` | `pd.qcut()` | `F.ntile(4)` |

Al final hay una guía de **cuándo usar cada herramienta**.

## Preparación

- `leer()` carga una tabla Gold y convierte las columnas `DECIMAL` a `double`. Así pandas las trata como números normales (sin conversión, pandas las recibe como objetos `Decimal` y algunas operaciones fallan).
- **Buena práctica:** filtrar en Spark **antes** de `toPandas()`. `toPandas()` trae todos los datos a la memoria de una sola máquina; si se filtra primero, solo viaja lo necesario.

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

GOLD = "rd_economic_intelligence.gold"
PRINCIPALES = ["gasolina_premium", "gasolina_regular", "gasoil_optimo", "gasoil_regular", "glp", "kerosene"]

def leer(tabla):
    """Carga una tabla Gold y convierte DECIMAL a double para trabajar cómodo en pandas."""
    df = spark.table(f"{GOLD}.{tabla}")
    for columna, tipo in df.dtypes:
        if tipo.startswith("decimal"):
            df = df.withColumn(columna, F.col(columna).cast("double"))
    return df

print("Listo. Tablas Gold disponibles:", [t.name for t in spark.catalog.listTables(GOLD)])

## 1 · ¿Cuál fue la tasa promedio del dólar cada año desde 2020?

**Concepto:** agrupar y resumir. En SQL es `GROUP BY` con `AVG`, `MIN`, `MAX` y `COUNT`; en pandas, `groupby().agg()`; en PySpark, `groupBy().agg()`.

**Detalle de pandas:** `agg(tasa_promedio="mean", …)` crea varias columnas con nombre en una sola línea (*named aggregation*).

**Hallazgo:** el promedio anual pasó de RD$56.58 en 2020 a RD$61.99 en 2025; en 2026 va en RD$60.31.

In [0]:
# pandas
fx = (leer("exchange_rate_metrics")
      .filter("rate_type = 'sell' AND year(rate_date) >= 2020")   # filtrar en Spark antes de toPandas()
      .select("rate_date", "rate")
      .toPandas())

fx["anio"] = pd.to_datetime(fx["rate_date"]).dt.year

resumen_pd = (fx.groupby("anio")["rate"]
                .agg(tasa_promedio="mean", tasa_minima="min", tasa_maxima="max", dias_habiles="count")
                .round(2)
                .reset_index())
display(resumen_pd)

In [0]:
# PySpark
resumen_spark = (leer("exchange_rate_metrics")
                 .filter("rate_type = 'sell' AND year(rate_date) >= 2020")
                 .groupBy(F.year("rate_date").alias("anio"))
                 .agg(F.round(F.avg("rate"), 2).alias("tasa_promedio"),
                      F.round(F.min("rate"), 2).alias("tasa_minima"),
                      F.round(F.max("rate"), 2).alias("tasa_maxima"),
                      F.count("*").alias("dias_habiles"))
                 .orderBy("anio"))
display(resumen_spark)

## 2 · ¿Cuánto cambió la gasolina premium semana a semana?

**Concepto:** comparar con la fila anterior.
- **SQL:** `LAG(price_dop) OVER (ORDER BY period_start_date)`
- **pandas:** `shift(1)` mueve la columna una fila hacia abajo. **Hay que ordenar antes**: `shift` usa el orden actual del DataFrame.
- **PySpark:** `F.lag("price_dop").over(Window.orderBy(...))`. El orden va dentro de la ventana.

**Hallazgo:** de RD$338 a RD$353 en las últimas 10 semanas publicadas.

In [0]:
# pandas
gas = (leer("fuel_price_metrics")
       .filter("fuel_code = 'gasolina_premium'")
       .select("period_start_date", "price_dop")
       .toPandas()
       .sort_values("period_start_date"))                 # 1) ordenar de la semana más vieja a la más nueva

gas["precio_semana_anterior"] = gas["price_dop"].shift(1)  # 2) valor de la fila anterior
gas["cambio_dop"] = gas["price_dop"] - gas["precio_semana_anterior"]

display(gas.sort_values("period_start_date", ascending=False).head(10))

In [0]:
# PySpark
w = Window.orderBy("period_start_date")

gas_spark = (leer("fuel_price_metrics")
             .filter("fuel_code = 'gasolina_premium'")
             .select("period_start_date", "price_dop")
             .withColumn("precio_semana_anterior", F.lag("price_dop").over(w))
             .withColumn("cambio_dop", F.col("price_dop") - F.col("precio_semana_anterior"))
             .orderBy(F.desc("period_start_date"))
             .limit(10))
display(gas_spark)

## 3 · ¿Cuál es el precio más reciente de cada combustible?

**Concepto:** "el último registro de cada grupo".
- **SQL:** `ROW_NUMBER() OVER (PARTITION BY fuel_name ORDER BY fecha DESC)` y quedarse con `= 1`.
- **pandas:** ordenar de la fecha más nueva a la más vieja y tomar la primera fila de cada grupo con `groupby().head(1)`.
- **PySpark:** igual que SQL, con `F.row_number().over(Window.partitionBy(...).orderBy(...))`.

**Hallazgo:** kerosene RD$383 el más caro; GLP RD$135 el más barato (semana del 25-sep-2026).

In [0]:
# pandas
comb = (leer("fuel_price_metrics")
        .filter(F.col("fuel_code").isin(PRINCIPALES))
        .select("fuel_name", "period_start_date", "price_dop")
        .toPandas())

ultimo_pd = (comb.sort_values("period_start_date", ascending=False)  # lo más reciente primero
                 .groupby("fuel_name")
                 .head(1)                                         # primera fila de cada combustible
                 .sort_values("price_dop", ascending=False))
display(ultimo_pd)

In [0]:
# PySpark
w = Window.partitionBy("fuel_name").orderBy(F.desc("period_start_date"))

ultimo_spark = (leer("fuel_price_metrics")
                .filter(F.col("fuel_code").isin(PRINCIPALES))
                .withColumn("n", F.row_number().over(w))
                .filter("n = 1")
                .select("fuel_name", "period_start_date", "price_dop")
                .orderBy(F.desc("price_dop")))
display(ultimo_spark)

## 4 · ¿Qué distribuidora eléctrica tiene más pérdidas? (y RANK vs DENSE_RANK)

**Concepto:** ranking.
- **SQL:** `RANK()` (1, 1, 3) y `DENSE_RANK()` (1, 1, 2).
- **pandas:** `rank(method="min")` equivale a `RANK`, `rank(method="dense")` a `DENSE_RANK` y `rank(method="first")` a `ROW_NUMBER`. Con `ascending=False`, el valor más alto queda en el puesto 1.
- **PySpark:** `F.rank()` y `F.dense_rank()` sobre una ventana ordenada.

Primero, un ejemplo con empates para ver la diferencia; después, los datos reales.

**Hallazgo (julio 2026):** Edeeste pierde 60.2 % de la energía que compra; Edesur 36.2 %; Edenorte 30.2 %.

In [0]:
# Ejemplo con empates: los tres métodos de pandas
notas = pd.DataFrame({"estudiante": ["Ana", "Luis", "Marta", "Pedro", "Rosa"],
                      "nota":       [95,    90,     90,      85,      80]})
notas["row_number (first)"] = notas["nota"].rank(method="first", ascending=False).astype(int)
notas["rank (min)"]         = notas["nota"].rank(method="min",   ascending=False).astype(int)
notas["dense_rank (dense)"] = notas["nota"].rank(method="dense", ascending=False).astype(int)
display(notas)

In [0]:
# pandas: datos reales
elec = leer("electricity_metrics").filter("distributor <> 'Total EDE'")
ultimo_mes = elec.agg(F.max("reference_month")).first()[0]

perdidas = (elec.filter(F.col("reference_month") == ultimo_mes)
                .select("distributor", "reference_month", "losses_pct")
                .toPandas())
perdidas["puesto_rank"]       = perdidas["losses_pct"].rank(method="min",   ascending=False).astype(int)
perdidas["puesto_dense_rank"] = perdidas["losses_pct"].rank(method="dense", ascending=False).astype(int)
display(perdidas.sort_values("puesto_rank").round(1))

In [0]:
# PySpark
w = Window.orderBy(F.desc("losses_pct"))

perdidas_spark = (leer("electricity_metrics")
                  .filter("distributor <> 'Total EDE'")
                  .filter(F.col("reference_month") == ultimo_mes)
                  .select("distributor", "reference_month", F.round("losses_pct", 1).alias("losses_pct"))
                  .withColumn("puesto_rank", F.rank().over(w))
                  .withColumn("puesto_dense_rank", F.dense_rank().over(w))
                  .orderBy("puesto_rank"))
display(perdidas_spark)

## 5 · ¿Qué porcentaje de la energía comprada se lleva cada distribuidora?

**Concepto:** participación sobre el total, sin perder el detalle de cada fila.
- **SQL:** `x / SUM(x) OVER ()`
- **pandas:** `df["x"] / df["x"].sum()`. En pandas es directo, porque `sum()` devuelve un número que se divide en cada fila.
- **PySpark:** `F.sum("x").over(Window.partitionBy())`: una ventana sin columnas es toda la tabla.

**Hallazgo (julio 2026):** Edeeste 37.4 %, Edesur 33.0 %, Edenorte 29.6 % de 2,132 GWh.

In [0]:
# pandas
energia = (leer("electricity_metrics")
           .filter("distributor <> 'Total EDE'")
           .filter(F.col("reference_month") == ultimo_mes)
           .select("distributor", "energy_bought_gwh")
           .toPandas())
energia["participacion_pct"] = energia["energy_bought_gwh"] / energia["energy_bought_gwh"].sum() * 100
display(energia.sort_values("participacion_pct", ascending=False).round(1))

In [0]:
# PySpark
total = Window.partitionBy()   # ventana vacía = toda la tabla

energia_spark = (leer("electricity_metrics")
                 .filter("distributor <> 'Total EDE'")
                 .filter(F.col("reference_month") == ultimo_mes)
                 .select("distributor", "energy_bought_gwh")
                 .withColumn("participacion_pct",
                             F.round(F.col("energy_bought_gwh") / F.sum("energy_bought_gwh").over(total) * 100, 1))
                 .orderBy(F.desc("participacion_pct")))
display(energia_spark)

## 6 · ¿Cuánta energía han comprado las EDE en 2026, mes a mes? (acumulado)

**Concepto:** total acumulado (*running total*).
- **SQL:** `SUM(x) OVER (ORDER BY mes ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`
- **pandas:** `cumsum()`, después de ordenar por mes.
- **PySpark:** `F.sum("x").over(Window.orderBy("mes").rowsBetween(Window.unboundedPreceding, Window.currentRow))`

**Hallazgo:** 12,072 GWh de enero a julio de 2026.

In [0]:
# pandas
ede = (leer("electricity_metrics")
       .filter("distributor = 'Total EDE' AND year(reference_month) = 2026")
       .select("reference_month", "energy_bought_gwh")
       .toPandas()
       .sort_values("reference_month"))
ede["acumulado_gwh"] = ede["energy_bought_gwh"].cumsum()
display(ede.round(1))

In [0]:
# PySpark
acumulado = Window.orderBy("reference_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

ede_spark = (leer("electricity_metrics")
             .filter("distributor = 'Total EDE' AND year(reference_month) = 2026")
             .select("reference_month", F.round("energy_bought_gwh", 1).alias("energy_bought_gwh"))
             .withColumn("acumulado_gwh", F.round(F.sum("energy_bought_gwh").over(acumulado), 1))
             .orderBy("reference_month"))
display(ede_spark)

## 7 · ¿Cuál es el promedio móvil de 3 meses de la inflación?

**Concepto:** promedio móvil.
- **SQL:** `AVG(x) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)`
- **pandas:** `rolling(window=3, min_periods=1).mean()`. `min_periods=1` hace que los primeros meses promedien con los que haya, igual que SQL.
- **PySpark:** `F.avg("x").over(Window.orderBy("mes").rowsBetween(-2, 0))`. `-2` son dos filas antes y `0` la fila actual.

**Detalle:** se calcula sobre **toda la serie** y se filtra al final. Si se filtra antes, el primer mes mostrado pierde los meses anteriores para su promedio (en SQL, esto se resuelve con `QUALIFY`).

**Hallazgo:** de 3.90 % (oct-2025) a 5.13 % (oct-2026).

In [0]:
# pandas
ipc = (leer("inflation_metrics")
       .filter("series_type = 'headline'")
       .select("reference_month", "yoy_inflation_pct")
       .toPandas()
       .sort_values("reference_month"))
ipc["promedio_movil_3m"] = ipc["yoy_inflation_pct"].rolling(window=3, min_periods=1).mean()

display(ipc[pd.to_datetime(ipc["reference_month"]) >= "2025-10-01"].round(2))   # filtrar después de calcular

In [0]:
# PySpark
movil = Window.orderBy("reference_month").rowsBetween(-2, 0)

ipc_spark = (leer("inflation_metrics")
             .filter("series_type = 'headline'")
             .select("reference_month", "yoy_inflation_pct")
             .withColumn("promedio_movil_3m", F.avg("yoy_inflation_pct").over(movil))
             .filter("reference_month >= '2025-10-01'")                                # filtrar después de calcular
             .select("reference_month", F.round("yoy_inflation_pct", 2).alias("yoy_inflation_pct"),
                     F.round("promedio_movil_3m", 2).alias("promedio_movil_3m"))
             .orderBy("reference_month"))
display(ipc_spark)

## 8 · ¿Qué productos agrícolas están en el 25 % que más subió?

**Concepto:** dividir en 4 grupos del mismo tamaño (cuartiles).
- **SQL:** `NTILE(4) OVER (ORDER BY alza DESC)`
- **pandas:** `pd.qcut(…, q=4, labels=[1, 2, 3, 4])`. `qcut` corta por cuantiles; para que se comporte igual que `NTILE` (grupos por cantidad de filas, aunque haya empates) se aplica sobre `rank(method="first")`.
- **PySpark:** `F.ntile(4).over(Window.orderBy(F.desc("alza")))`

**Ojo con la diferencia:** 91 productos no se dividen exacto entre 4. `NTILE` reparte las filas sobrantes en los primeros grupos (23, 23, 23, 22) y `qcut` puede ponerlas en otro (23, 23, 22, 23). El primer cuartil es el mismo en los dos, pero los grupos 3 y 4 pueden variar en un producto. Es un buen detalle para comentar en una entrevista.

**Hallazgo (octubre 2026, minorista):** de 91 productos, los 23 del primer cuartil subieron entre 9.5 % y 56.6 % en un año.

In [0]:
# pandas
agri = leer("agri_price_monthly")
mes_agri = agri.agg(F.max("reference_month")).first()[0]

alzas = (agri.filter((F.col("reference_month") == mes_agri) & (F.col("channel") == "retail")
                     & (F.col("market") == "Promedio mercados") & F.col("yoy_pct").isNotNull())
             .select("rubro", "product", F.round("yoy_pct", 2).alias("alza_pct"))
             .toPandas())

orden = alzas["alza_pct"].rank(method="first", ascending=False)      # 1 = el que más subió
alzas["cuartil"] = pd.qcut(orden, q=4, labels=[1, 2, 3, 4]).astype(int)

display(alzas.groupby("cuartil")["alza_pct"].agg(productos="count", alza_minima="min",
                                                 alza_maxima="max", alza_promedio="mean").round(2).reset_index())
display(alzas[alzas["cuartil"] == 1].sort_values("alza_pct", ascending=False))

In [0]:
# PySpark
w = Window.orderBy(F.desc("alza_pct"))

alzas_spark = (agri.filter((F.col("reference_month") == mes_agri) & (F.col("channel") == "retail")
                           & (F.col("market") == "Promedio mercados") & F.col("yoy_pct").isNotNull())
                   .select("rubro", "product", F.round("yoy_pct", 2).alias("alza_pct"))
                   .withColumn("cuartil", F.ntile(4).over(w)))

display(alzas_spark.groupBy("cuartil")
                   .agg(F.count("*").alias("productos"), F.min("alza_pct").alias("alza_minima"),
                        F.max("alza_pct").alias("alza_maxima"), F.round(F.avg("alza_pct"), 2).alias("alza_promedio"))
                   .orderBy("cuartil"))
display(alzas_spark.filter("cuartil = 1").orderBy(F.desc("alza_pct")))

## Comprobación: ¿pandas y PySpark dan lo mismo?

Una buena costumbre es verificar que dos caminos distintos lleguen al mismo resultado. Esta celda compara la pregunta 1 en ambas versiones.

In [0]:
a = resumen_pd.sort_values("anio").reset_index(drop=True)
b = resumen_spark.toPandas().sort_values("anio").reset_index(drop=True)

iguales = (a["tasa_promedio"].round(2).tolist() == b["tasa_promedio"].astype(float).round(2).tolist()
           and a["dias_habiles"].tolist() == b["dias_habiles"].tolist())
print("pandas y PySpark coinciden:", iguales)
assert iguales, "Los resultados no coinciden"

## ¿Cuándo usar cada herramienta?

| | SQL | pandas | PySpark |
|---|---|---|---|
| **Dónde corre** | Motor SQL (SQL Warehouse, Fabric, SQL Server) | Memoria de **una sola máquina** | **Varias máquinas** en paralelo (cluster) |
| **Tamaño de datos** | Cualquiera | Hasta lo que quepa en memoria (miles o pocos millones de filas) | Millones o miles de millones de filas |
| **Ideal para** | Consultas, reportes, vistas para Power BI | Análisis exploratorio, gráficos, datos ya resumidos | Pipelines ETL, capas Bronze / Silver / Gold |
| **En este proyecto** | Notebook `09_SQL_Analytics`, validaciones | Este notebook, con datos ya filtrados | Todos los notebooks Silver y Gold |

**Regla práctica:** transformar en PySpark o SQL mientras los datos son grandes; pasar a pandas (`toPandas()`) solo cuando ya están filtrados o resumidos. Este flujo es el mismo en **Databricks**, en los notebooks de **Microsoft Fabric** y en **Azure Synapse**, porque los tres usan Spark y pandas.